In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
path = '/content/drive/MyDrive/yoga-posture-dl'
print(os.listdir(path))

['train_keypoints.csv', 'test_keypoints.csv', 'label_classes.npy', 'yoga_model.h5']


In [ ]:
!pip install mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
import mlflow
import mlflow.tensorflow

# ── Load data ──────────────────────────────────────────────────────────────
train_df = pd.read_csv('/content/drive/MyDrive/yoga-posture-dl/train_keypoints.csv')
test_df  = pd.read_csv('/content/drive/MyDrive/yoga-posture-dl/test_keypoints.csv')

feature_cols = [c for c in train_df.columns if c != 'label']
X_train = train_df[feature_cols].values.astype(np.float32)
X_test  = test_df[feature_cols].values.astype(np.float32)

le = LabelEncoder()
y_train = le.fit_transform(train_df['label'])
y_test  = le.transform(test_df['label'])
np.save('/content/drive/MyDrive/yoga-posture-dl/label_classes.npy', le.classes_)

n_classes = len(le.classes_)
print(f"{n_classes} pose classes | {X_train.shape[1]} features | {len(X_train)} train samples")

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

# ── Model ──────────────────────────────────────────────────────────────────
def build_model(input_dim, n_classes, dropout=0.4, units=256):
    inp = keras.Input(shape=(input_dim,))
    x = keras.layers.Dense(units, activation='relu')(inp)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(dropout)(x)
    x = keras.layers.Dense(units, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(dropout)(x)
    x = keras.layers.Dense(units // 2, activation='relu')(x)
    x = keras.layers.Dropout(dropout / 2)(x)
    out = keras.layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inp, out)

# ── Train ──────────────────────────────────────────────────────────────────
mlflow.set_experiment("yoga-posture-detection")
PARAMS = dict(lr=1e-3, dropout=0.4, units=256, epochs=80, batch_size=64)

with mlflow.start_run():
    mlflow.log_params(PARAMS)
    model = build_model(X_train.shape[1], n_classes, PARAMS['dropout'], PARAMS['units'])
    model.compile(
        optimizer=keras.optimizers.Adam(PARAMS['lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, verbose=1),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=PARAMS['epochs'],
        batch_size=PARAMS['batch_size'],
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    val_acc = max(history.history['val_accuracy'])
    mlflow.log_metric("best_val_accuracy", val_acc)
    print(f"\nBest val accuracy: {val_acc:.4f}")

    model.save('/content/drive/MyDrive/yoga-posture-dl/yoga_model.h5')
    print("Model saved to Drive.")

107 pose classes | 147 features | 4415 train samples


2026/04/11 13:13:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/11 13:13:13 INFO mlflow.store.db.utils: Updating database tables
2026/04/11 13:13:15 INFO mlflow.tracking.fluent: Experiment with name 'yoga-posture-detection' does not exist. Creating a new experiment.


Epoch 1/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 9s 56ms/step - accuracy: 0.0399 - loss: 4.6673 - val_accuracy: 0.0716 - val_loss: 4.3919 - learning_rate: 0.0010
Epoch 2/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1325 - loss: 3.7043 - val_accuracy: 0.1637 - val_loss: 3.4824 - learning_rate: 0.0010
Epoch 3/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1941 - loss: 3.2785 - val_accuracy: 0.2779 - val_loss: 2.8692 - learning_rate: 0.0010
Epoch 4/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2308 - loss: 3.0338 - val_accuracy: 0.3606 - val_loss: 2.5902 - learning_rate: 0.0010
Epoch 5/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2627 - loss: 2.8461 - val_accuracy: 0.3922 - val_loss: 2.4248 - learning_rate: 0.0010
Epoch 6/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2734 - loss: 2.7477 - val_accuracy: 0.4450 - val_loss: 2.3115 - learning_rate: 0.0010
Epoch 7/80
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3060 - loss: 2.6267 - val_accuracy:


Best val accuracy: 0.6684
Model saved to Drive.


In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yoga-posture-dl/yoga_model.h5')
files.download('/content/drive/MyDrive/yoga-posture-dl/label_classes.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.save('/content/drive/MyDrive/yoga-posture-dl/yoga_model.keras')

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yoga-posture-dl/yoga_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>